In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv

import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 43
        environment_string = "mimic"
        training_timesteps = 100_000
        num_concepts_selected = 70
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = True
        run_iterative = False
        run_two_stage = False 
        run_imperfect=False 
        # Experiment #3
        cbm_accuracy_by_concept = None
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 3
        selections_per_round = 1
        initial_concepts = 1
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [6]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 43, 'environment_string': 'mimic', 'training_timesteps': 100000, 'selection_function': 'q_value', 'num_concepts_selected': 70, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 3, 'selections_per_round': 1, 'initial_concepts': 1, 'run_basic': True, 'run_iterative': False, 'run_two_stage': False}


In [7]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

### Basic Setup

In [8]:
if is_main:
    concept_list = get_concepts(environment_string,concept_source,seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   

In [9]:
from gym import RewardWrapper
class ClipRewardEnv(RewardWrapper):
    def __init__(self, env, min_reward=-15, max_reward=15):
        super().__init__(env)
        self.min_reward = min_reward
        self.max_reward = max_reward

    def reward(self, reward):
        return np.clip(reward, self.min_reward, self.max_reward)
env = DummyVecEnv([lambda: ground_truth_env])  # Wrap your env
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_reward=10.0)


In [11]:
groundtruth_model = train_ppo_model(env,environment_string+"_raw",total_timesteps=training_timesteps,policy="MlpPolicy")

Running MIMIC RAW
Step: 10112 | AvgR: 10.535 | EV: 0.036319196224212646 | VLoss: 0.4559535626322031 | KL: 0.01798689365386963 | ClipF: 0.138671875 | GradN: 0.4999996931029649
Step: 20224 | AvgR: 11.729 | EV: -0.4968606233596802 | VLoss: 0.04637918609660119 | KL: 0.026059500873088837 | ClipF: 0.1484375 | GradN: 0.49999962479335297
Step: 30336 | AvgR: 14.135 | EV: -0.3610037565231323 | VLoss: 0.14124974282458425 | KL: 0.012116577476263046 | ClipF: 0.078125 | GradN: 0.49999943325016494
Step: 40448 | AvgR: 13.649 | EV: -0.22423923015594482 | VLoss: 0.048535577836446464 | KL: 0.024071794003248215 | ClipF: 0.130859375 | GradN: 0.4999994042385734
Step: 50560 | AvgR: 12.077 | EV: -0.6750531196594238 | VLoss: 0.0812184326350689 | KL: 0.01636732742190361 | ClipF: 0.111328125 | GradN: 0.499999454504215
Step: 60672 | AvgR: 12.758 | EV: 0.0741504430770874 | VLoss: 0.101791609544307 | KL: 0.021314920857548714 | ClipF: 0.1171875 | GradN: 0.49999918269810134
Step: 70784 | AvgR: 12.671 | EV: 0.01691639

In [24]:
model_random = RandomAgent(GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])))

In [30]:
model_full_concepts = model 

In [12]:
MIMICraw = pd.read_csv("../../data/mimic_github/ai_clinician/data/mimic_model/train/MIMICraw.csv")
metadata = pd.read_csv("../../data/mimic_github/ai_clinician/data/mimic_model/train/metadata.csv")
MIMICzs = pd.read_csv("../../data/mimic_github/ai_clinician/data/mimic_model/train/MIMICzs.csv")

C_ICUSTAYID = "icustayid"
unique_icu_stays = metadata[C_ICUSTAYID].unique()

n_action_bins = 5
gamma = 0.99

all_actions, _, _ = fit_action_bins(
    MIMICraw[C_INPUT_STEP],
    MIMICraw[C_MAX_DOSE_VASO],
    n_action_bins=n_action_bins
)

train_ids, val_ids = train_test_split(unique_icu_stays, test_size=0.1,random_state=seed+3)
train_indexes = metadata[metadata[C_ICUSTAYID].isin(train_ids)].index
val_indexes = metadata[metadata[C_ICUSTAYID].isin(val_ids)].index

X_train = MIMICzs.iloc[train_indexes]
X_val = MIMICzs.iloc[val_indexes]

metadata_val = metadata.iloc[val_indexes]
actions_val = all_actions[val_indexes]

states_train = additional_info['clusterer'].predict(X_train.values)
states_val = additional_info['clusterer'].predict(X_val.values)

phys_probs = compute_physician_probabilities(additional_info['physpol'],np.max(states_train)+1,states=states_val, actions=actions_val)
model_probs = compute_model_probabilities(model,additional_info['concept_list'],states=states_val, actions=actions_val)


/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:411: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()


In [13]:
np.mean(np.abs(phys_probs-model_probs))

0.25545118355518637

In [50]:
val_bootwis, _,  _ = evaluate_policy_wis(
        metadata_val,
        phys_probs,
        model_probs,
        [15,-15],
        gamma,
        200
    )
val_bootwis

WIS estimation:   4%|▍         | 9/200 [00:00<00:04, 44.77it/s]

ESS: 1.0070968964045879  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 1.76749471e+06
 1.22957313e+06 8.82618403e+05 9.53722464e+04 4.31316699e+04
 4.09065929e+04 3.50401753e+04]
Top weights fraction of total: 0.9999999997247698
Cumulative fraction of total weights (top 1..10): [0.99646406 0.99999853 0.99999999 0.99999999 1.         1.
 1.         1.         1.         1.        ]
ESS: 1.0070968987285556  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 1.76749471e+06
 1.22957313e+06 8.82618403e+05 5.45950504e+05 4.09065929e+04
 3.50401753e+04 2.43749551e+04]
Top weights fraction of total: 0.9999999997869924
Cumulative fraction of total weights (top 1..10): [0.99646406 0.99999853 0.99999999 0.99999999 1.         1.
 1.         1.         1.         1.        ]
ESS: 1.0000029914869435  / N: 1629
Top weights: [3.53855494e+14 5.17797030e+08 8.01915166e+06 1.76749471e+06
 8.82618403e+05 5.45950504e+05 9.53722464e+04 4.09065929e+04
 3.50401753e+04 

WIS estimation:   7%|▋         | 14/200 [00:00<00:04, 46.01it/s]

ESS: 1.0070969313586005  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 8.01915166e+06
 1.22957313e+06 8.82618403e+05 9.53722464e+04 4.31316699e+04
 2.43749551e+04 1.10578378e+04]
Top weights fraction of total: 0.9999999998616231
Cumulative fraction of total weights (top 1..10): [0.99646404 0.99999851 0.99999997 0.99999999 1.         1.
 1.         1.         1.         1.        ]
ESS: 1.0008438177362478  / N: 1629
Top weights: [1.25512985e+12 5.17797030e+08 8.01915166e+06 1.76749471e+06
 1.22957313e+06 5.45950504e+05 4.09065929e+04 3.50401753e+04
 2.43749551e+04 1.67541931e+04]
Top weights fraction of total: 0.9999999447669567
Cumulative fraction of total weights (top 1..10): [0.99957827 0.99999064 0.99999703 0.99999844 0.99999942 0.99999985
 0.99999988 0.99999991 0.99999993 0.99999994]
ESS: 1.0008425908435037  / N: 1629
Top weights: [1.25512985e+12 5.17797030e+08 8.01915166e+06 1.22957313e+06
 8.82618403e+05 5.45950504e+05 9.53722464e+04 4.31316699e+04
 4.09065

WIS estimation:  12%|█▏        | 24/200 [00:00<00:03, 46.94it/s]

...

WIS estimation:  99%|█████████▉| 198/200 [00:04<00:00, 43.51it/s]

ESS: 1.0008451775377132  / N: 1629
Top weights: [1.25512985e+12 5.17797030e+08 8.01915166e+06 1.76749471e+06
 1.22957313e+06 8.82618403e+05 5.45950504e+05 4.09065929e+04
 3.50401753e+04 2.43749551e+04]
Top weights fraction of total: 0.9999999550087996
Cumulative fraction of total weights (top 1..10): [0.99957759 0.99998996 0.99999635 0.99999776 0.99999874 0.99999944
 0.99999988 0.99999991 0.99999994 0.99999996]
ESS: 1.007096939337651  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 8.01915166e+06
 1.76749471e+06 1.22957313e+06 5.45950504e+05 4.31316699e+04
 4.09065929e+04 2.43749551e+04]
Top weights fraction of total: 0.9999999997449327
Cumulative fraction of total weights (top 1..10): [0.99646404 0.99999851 0.99999997 0.99999999 0.99999999 1.
 1.         1.         1.         1.        ]
ESS: 1.0070968925850499  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 1.76749471e+06
 8.82618403e+05 5.45950504e+05 9.53722464e+04 4.31316699e+04
 4.090659

WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 44.32it/s]

ESS: 1.00709400461906  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 8.01915166e+06 1.76749471e+06
 1.22957313e+06 8.82618403e+05 9.53722464e+04 4.31316699e+04
 3.50401753e+04 2.43749551e+04]
Top weights fraction of total: 0.9999999998394674
Cumulative fraction of total weights (top 1..10): [0.99646549 0.99999997 0.99999999 0.99999999 1.         1.
 1.         1.         1.         1.        ]
ESS: 1.0070969447559233  / N: 1629
Top weights: [3.53855494e+14 1.25512985e+12 5.17797030e+08 8.01915166e+06
 1.76749471e+06 1.22957313e+06 8.82618403e+05 5.45950504e+05
 9.53722464e+04 4.31316699e+04]
Top weights fraction of total: 0.9999999996250947
Cumulative fraction of total weights (top 1..10): [0.99646404 0.99999851 0.99999996 0.99999999 0.99999999 1.
 1.         1.         1.         1.        ]


array([ 12.42503097,  12.42503097,  12.51770643,  12.42503097,
        12.42503097,  12.42503083,  12.42503097, -13.70269578,
        12.42503097,  12.42503097, -13.69169874, -13.69171488,
       -13.69169673,  12.42503097,  12.42503097,  12.42503097,
        12.42503097,  12.42503097,  12.51770643,  12.42503097,
        12.42503097,  12.51770643, -13.70267698, -13.70272206,
        12.42503084,  12.51770643,  12.51770643,  12.51770643,
       -13.70272157, -13.6916782 ,  12.51770643,  12.42503083,
        12.51770643,  12.51770642,  12.79973893, -13.70265801,
        12.42503097,  12.7330812 ,  12.42503098,  12.42503097,
        12.51770643,  12.51770643,  12.51770643,  12.42503097,
        12.52081861,  12.42503097,  12.42503098, -13.70269481,
        12.42503083,  12.42503097, -13.69172425,  12.42503097,
        12.51770643,  12.42503084,  12.42503097,  12.42503097,
       -13.70255419,  12.42503098, -13.69184553,  12.51770643,
        12.42503083,  12.42503097, -13.69173684, -13.69

In [48]:
np.mean(val_bootwis)

7.62112346208815

In [ ]:
groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed+s)


In [38]:
env, eval_env, additional_info = get_environment(environment_string,concept_list[0:1],seed)
model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

Step: 10240 | AvgR: 4.490 | EV: 0.0012395977973937988 | VLoss: 54.591317653656006 | KL: 0.002177231712266803 | ClipF: 0.00927734375 | GradN: 0.4999998957072498
Step: 20480 | AvgR: 14.910 | EV: 0.01335078477859497 | VLoss: 83.77965927124023 | KL: 0.00032640062272548676 | ClipF: 0.0 | GradN: 0.4999999298947779
Step: 30720 | AvgR: 10.646 | EV: 0.14259254932403564 | VLoss: 126.78786945343018 | KL: 0.0002701166085898876 | ClipF: 0.0 | GradN: 0.49999999355242614
Step: 40960 | AvgR: 12.430 | EV: -0.0003254413604736328 | VLoss: 49.432279109954834 | KL: 0.009957745671272278 | ClipF: 0.0703125 | GradN: 0.4999999705331464
Step: 51200 | AvgR: 12.222 | EV: -0.05017554759979248 | VLoss: 53.82296943664551 | KL: 0.0007416083244606853 | ClipF: 0.0 | GradN: 0.5000000071680414
Step: 61440 | AvgR: 9.846 | EV: 0.16108500957489014 | VLoss: 122.2431755065918 | KL: 0.0010140163358300924 | ClipF: 0.0 | GradN: 0.49999997817788844
Step: 71680 | AvgR: 11.553 | EV: 0.023051679134368896 | VLoss: 73.88174343109131 |

In [9]:
if is_main:
    # model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,training_timesteps,seed)
    
    # if os.path.exists(model_name):
    #     groundtruth_model = PPO.load(model_name)
    # else:
    if "cyclic" in environment_string or "tree" in environment_string or "mimic" in environment_string:
        policy = "MlpPolicy"
    else:
        policy = "CnnPolicy"
    if environment_string == "mimic":
        groundtruth_model = train_ppo_model(ground_truth_env,environment_string+"_raw",total_timesteps=training_timesteps,policy=policy)
    else:
        groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=training_timesteps,policy=policy)

    # # groundtruth_model.save(model_name)
    groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed+s)
    # # results['ground_truth'] = {'reward':groundtruth_reward}
    # print(groundtruth_reward)

Running MIMIC RAW
Step: 10112 | AvgR: 12.285 | EV: 0.09988749027252197 | VLoss: 33.54427766799927 | KL: 0.013752548955380917 | ClipF: 0.09375 | GradN: 0.49999984338608633
Step: 20224 | AvgR: 12.421 | EV: 0.03202420473098755 | VLoss: 68.87376642227173 | KL: 0.018605949357151985 | ClipF: 0.16015625 | GradN: 0.5000000226539344
Step: 30336 | AvgR: 15.619 | EV: -0.2592536211013794 | VLoss: 55.60432839393616 | KL: 0.009234948083758354 | ClipF: 0.0625 | GradN: 0.500000015891949
Step: 40448 | AvgR: 11.913 | EV: -0.005817532539367676 | VLoss: 51.32333850860596 | KL: 0.0223804023116827 | ClipF: 0.140625 | GradN: 0.4999998452425185
Step: 50560 | AvgR: 13.303 | EV: 0.04461216926574707 | VLoss: 40.18546736240387 | KL: 0.0691666305065155 | ClipF: 0.236328125 | GradN: 0.49999994196974157
Step: 60672 | AvgR: 12.158 | EV: -0.05541181564331055 | VLoss: 46.82662761211395 | KL: 0.02257823571562767 | ClipF: 0.181640625 | GradN: 0.4999999049698363
Step: 70784 | AvgR: 11.793 | EV: 0.017477035522460938 | VLos

NameError: name 's' is not defined

In [42]:
for s in range(3):
    groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed+s)
    # results['ground_truth'] = {'reward':groundtruth_reward}
    print(groundtruth_reward)

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:394: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 44.02it/s]


-6.466338778735175


/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:394: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 43.58it/s]


12.671811506142552


/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:394: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 42.91it/s]

8.222532348734507


### Basic Comparison

In [10]:
if is_main:    
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,training_timesteps,seed,selection_function,concept_source)
    results['basic_comparison'] = {}
    # if os.path.exists(model_name):
    #     q_estimates = pickle.load(open(model_name,"rb"))
    # else:
    if selection_function == "q_value":
        if environment_string == "mimic":
            modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                        for concept in concept_list]

            q_estimates = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,learning_rate=1e-2,mimic=True,total_timesteps=10000,final_training=0)
        else:
            q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    elif selection_function == "policy":
        if environment_string == "mimic":
            modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                        for concept in concept_list]

            q_estimates = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,mimic=True)
        else:
            q_estimates = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)
    pickle.dump(q_estimates,open(model_name,"wb"))

Starting stable training for sparse rewards...
Loss mean 0


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.transitions to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.transitions` for environment variables or `env.get_wrapper_attr('transitions')` that will search the reminding wrappers.
  logger.warn(
/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/env_utils.py:373: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:274.)
  states = torch.FloatTensor(states)


Episode 25, Avg Reward: 13.76, Loss (mean/std/max): 1.10/0.92/5.27, Epsilon: 0.099
Episode 50, Avg Reward: 12.89, Loss (mean/std/max): 0.99/0.74/5.27, Epsilon: 0.098
Loss mean 0.2497495211660862
Final training phase...
Training completed. Final epsilon: 0.078
Final average reward: 16.39
Final loss stats - Mean: 0.16, Std: 0.10, Max: 0.46
Total state-action pairs: 580


In [38]:
if is_main and run_basic:
    for s in range(3):
        # Train a random policy
        if environment_string == "mimic":
            model = RandomAgent(GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])))
        else:
            model = RandomAgent(ground_truth_gym_env)
        random_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,model,seed+s)
        results['basic_comparison']['random'] = {'reward':random_reward}
        print(results['basic_comparison']['random']['reward'])

WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 46.32it/s]


-7.3837926017942666


WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 43.30it/s]


9.194946740943205


WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 43.82it/s]

11.787814466956196


In [26]:
if is_main and run_basic:
    # Train a random selector
    subset_concept, random_idx = random_selection(concept_list,num_concepts_selected)
    subset_concept = [concept_list[i] for i in random_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['random_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
    print(results['basic_comparison']['random_selection']['reward'])

Step: 10240 | AvgR: 10.629 | EV: 0.28657233715057373 | VLoss: 49.70472049713135 | KL: 0.003087896853685379 | ClipF: 0.00439453125 | GradN: 0.49999999803548606
Step: 20480 | AvgR: 12.631 | EV: -0.0711902379989624 | VLoss: 61.29444217681885 | KL: 0.020730264484882355 | ClipF: 0.0947265625 | GradN: 0.49999998729211814
Step: 30720 | AvgR: 12.539 | EV: 0.021690785884857178 | VLoss: 48.28668832778931 | KL: 0.012140648439526558 | ClipF: 0.07958984375 | GradN: 0.49999989061631317
Step: 40960 | AvgR: 12.349 | EV: 0.033607423305511475 | VLoss: 48.897249698638916 | KL: 0.019303761422634125 | ClipF: 0.1123046875 | GradN: 0.4999999705748365
Step: 51200 | AvgR: 13.837 | EV: -0.026220321655273438 | VLoss: 51.74045753479004 | KL: 0.01510784961283207 | ClipF: 0.1689453125 | GradN: 0.49999997350071695
Step: 61440 | AvgR: 12.401 | EV: -0.01706695556640625 | VLoss: 59.391411781311035 | KL: 0.011132177896797657 | ClipF: 0.05517578125 | GradN: 0.4999999588687203
Step: 71680 | AvgR: 12.924 | EV: 0.2440967559

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:394: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 46.53it/s]

9.504264399939398


In [11]:
if is_main and run_basic:
    # Train a greedy selector
    subset_concept, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy'] = {'concepts': greedy_idx, 'reward':greedy_selection_reward}
    print(results['basic_comparison']['greedy']['reward'])


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/numpy/core/_methods.py:223: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.8/site-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Step: 10240 | AvgR: 10.334 | EV: -0.05010783672332764 | VLoss: 29.85785746574402 | KL: 0.028909973800182343 | ClipF: 0.2109375 | GradN: 0.49999995409949194
Step: 20480 | AvgR: 12.057 | EV: -0.06533479690551758 | VLoss: 43.189356327056885 | KL: 0.023626402020454407 | ClipF: 0.12060546875 | GradN: 0.5000000020697605
Step: 30720 | AvgR: 12.724 | EV: -0.07229757308959961 | VLoss: 46.94076633453369 | KL: 0.008230289444327354 | ClipF: 0.09033203125 | GradN: 0.49999994489694305
Step: 40960 | AvgR: 11.698 | EV: -0.051131367683410645 | VLoss: 46.99556636810303 | KL: 0.010617172345519066 | ClipF: 0.0703125 | GradN: 0.4999999629359854
Step: 51200 | AvgR: 13.031 | EV: -0.04052853584289551 | VLoss: 55.440584659576416 | KL: 0.002489957259967923 | ClipF: 0.0078125 | GradN: 0.4999999299818776
Step: 61440 | AvgR: 9.931 | EV: -0.055008530616760254 | VLoss: 31.69535732269287 | KL: 0.023094475269317627 | ClipF: 0.1650390625 | GradN: 0.49999992025811185
Step: 71680 | AvgR: 13.612 | EV: 0.01231241226196289 

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:411: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation:   5%|▌         | 10/200 [00:00<00:03, 48.02it/s]

ESS: 1.6943674108209419  / N: 1629
Top weights: [1.58434099e+10 5.70995925e+09 2.86337728e+08 3.40130522e+07
 1.92302276e+07 1.55876567e+07 4.89941065e+06 3.90632597e+06
 3.54866618e+06 8.94717695e+05]
Top weights fraction of total: 0.9998656739141862
Cumulative fraction of total weights (top 1..10): [0.7226273  0.98306192 0.99612195 0.99767331 0.99855041 0.99926137
 0.99948484 0.99966301 0.99982487 0.99986567]
ESS: 1.122334858713395  / N: 1629
Top weights: [5.70995925e+09 2.86337728e+08 3.40130522e+07 1.92302276e+07
 4.89941065e+06 8.94717695e+05 5.43145758e+05 2.83239737e+05
 2.51179515e+05 2.30665853e+05]
Top weights fraction of total: 0.999958604759987
Cumulative fraction of total weights (top 1..10): [0.94272079 0.98999547 0.99561106 0.998786   0.9995949  0.99974261
 0.99983229 0.99987905 0.99992052 0.9999586 ]
ESS: 1.044858853970061  / N: 1629
Top weights: [1.58434099e+10 2.86337728e+08 3.40130522e+07 1.92302276e+07
 4.89941065e+06 3.90632597e+06 3.54866618e+06 7.75366535e+05
 5.

WIS estimation:   8%|▊         | 15/200 [00:00<00:03, 47.83it/s]

...

ESS: 1.129857074420527  / N: 1629
Top weights: [5.70995925e+09 2.86337728e+08 3.40130522e+07 1.92302276e+07
 1.55876567e+07 4.89941065e+06 3.54866618e+06 8.94717695e+05
 7.75366535e+05 5.71044982e+05]
Top weights fraction of total: 0.9997756033990781
Cumulative fraction of total weights (top 1..10): [0.9395737  0.98669057 0.99228741 0.99545175 0.99801669 0.99882289
 0.99940683 0.99955405 0.99968164 0.9997756 ]
ESS: 1.6476654520785852  / N: 1629
Top weights: [1.58434099e+10 5.70995925e+09 3.40130522e+07 1.55876567e+07
 4.89941065e+06 3.90632597e+06 3.54866618e+06 8.94717695e+05
 2.83239737e+05 2.51179515e+05]
Top weights fraction of total: 0.9999735213759932
Cumulative fraction of total weights (top 1..10): [0.73290332 0.99704142 0.99861483 0.99933591 0.99956255 0.99974325
 0.99990741 0.9999488  0.9999619  0.99997352]
ESS: 1.3238918160521347  / N: 1629
Top weights: [2.86337728e+08 3.40130522e+07 4.89941065e+06 3.54866618e+06
 8.94717695e+05 7.75366535e+05 5.71044982e+05 2.83239737e+05
 

WIS estimation:  95%|█████████▌| 190/200 [00:04<00:00, 46.68it/s]

ESS: 1.6890028628958567  / N: 1629
Top weights: [1.58434099e+10 5.70995925e+09 2.86337728e+08 3.40130522e+07
 4.89941065e+06 3.90632597e+06 3.54866618e+06 8.94717695e+05
 7.75366535e+05 5.71044982e+05]
Top weights fraction of total: 0.9999242867776735
Cumulative fraction of total weights (top 1..10): [0.72377476 0.98462293 0.9977037  0.99925752 0.99948134 0.99965979
 0.99982191 0.99986278 0.9998982  0.99992429]
ESS: 1.1283729754608922  / N: 1629
Top weights: [5.70995925e+09 2.86337728e+08 3.40130522e+07 1.92302276e+07
 1.55876567e+07 4.89941065e+06 8.94717695e+05 7.75366535e+05
 5.71044982e+05 2.83239737e+05]
Top weights fraction of total: 0.9998953853922904
Cumulative fraction of total weights (top 1..10): [0.94019157 0.98733942 0.99293995 0.99610636 0.998673   0.99947973
 0.99962705 0.99975472 0.99984875 0.99989539]
ESS: 1.0443302570869386  / N: 1629
Top weights: [1.58434099e+10 2.86337728e+08 3.40130522e+07 1.55876567e+07
 4.89941065e+06 3.90632597e+06 3.54866618e+06 8.94717695e+05


WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 46.25it/s]

ESS: 1.6478843491100832  / N: 1629
Top weights: [1.58434099e+10 5.70995925e+09 3.40130522e+07 1.55876567e+07
 4.89941065e+06 3.90632597e+06 3.54866618e+06 8.94717695e+05
 7.75366535e+05 5.43145758e+05]
Top weights fraction of total: 0.9999433708463182
Cumulative fraction of total weights (top 1..10): [0.73285464 0.99697519 0.9985485  0.99926953 0.99949616 0.99967685
 0.999841   0.99988238 0.99991825 0.99994337]
ESS: 1.6935025700437245  / N: 1629
Top weights: [1.58434099e+10 5.70995925e+09 2.86337728e+08 3.40130522e+07
 1.92302276e+07 1.55876567e+07 3.90632597e+06 3.54866618e+06
 8.94717695e+05 7.75366535e+05]
Top weights fraction of total: 0.9999328427250094
Cumulative fraction of total weights (top 1..10): [0.72281182 0.98331295 0.99637631 0.99792807 0.99880539 0.99951654
 0.99969475 0.99985665 0.99989747 0.99993284]
ESS: 1.1293595100540919  / N: 1629
Top weights: [5.70995925e+09 2.86337728e+08 3.40130522e+07 1.92302276e+07
 1.55876567e+07 4.89941065e+06 3.54866618e+06 7.75366535e+05


In [32]:
subset_concept, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

On selecting #0
On selecting #1
On selecting #2
On selecting #3
On selecting #4
On selecting #5
On selecting #6
On selecting #7
On selecting #8
On selecting #9
On selecting #10
On selecting #11
On selecting #12
On selecting #13
On selecting #14
On selecting #15
On selecting #16
On selecting #17
On selecting #18
On selecting #19
On selecting #20
On selecting #21
On selecting #22
On selecting #23
On selecting #24
On selecting #25
On selecting #26
On selecting #27
On selecting #28
On selecting #29
On selecting #30


KeyboardInterrupt: 

In [30]:
if is_main and run_basic:
    subset_concept, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,seed,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_iterative_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy_iterative'] = {'concepts': greedy_iterative_idx, 'reward': greedy_iterative_selection_reward}
    print(results['basic_comparison']['greedy_iterative']['reward'])

KeyboardInterrupt: 

In [36]:
if is_main and run_basic:
    subset_concept, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=100_000,policy="MlpPolicy")
    lp_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['lp'] = {'concepts': lp_idx, 'reward': lp_selection_reward}
    print(results['basic_comparison']['lp']['reward'])

On 0 168 168
On 1 8 8
On 2 14 14
On 3 10 10
On 4 12 12
On 5 55 55
On 6 10 10
On 7 11 11
On 8 8 8
On 9 5 5
On 10 48 48
On 11 13 13
On 12 16 16
On 13 12 12
On 14 15 15
On 15 285 285
On 16 13 13
On 17 12 12
On 18 11 11
On 19 13 13
On 20 16 16
On 21 11 11
On 22 11 11
On 23 8 8
On 24 11 11
Step: 10240 | AvgR: 10.184 | EV: 0.19019657373428345 | VLoss: 56.36835193634033 | KL: 0.00205355416983366 | ClipF: 0.00146484375 | GradN: 0.49999988666723016
Step: 20480 | AvgR: 10.039 | EV: -0.02176380157470703 | VLoss: 38.192848682403564 | KL: 0.0035497506614774466 | ClipF: 0.01025390625 | GradN: 0.499999972973612
Step: 30720 | AvgR: 9.527 | EV: 0.06672096252441406 | VLoss: 90.22645092010498 | KL: 0.0012427868787199259 | ClipF: 0.00048828125 | GradN: 0.49999997313570416
Step: 40960 | AvgR: 12.181 | EV: -0.05623626708984375 | VLoss: 55.93625020980835 | KL: 0.012289163656532764 | ClipF: 0.0537109375 | GradN: 0.4999999464871906
Step: 51200 | AvgR: 11.245 | EV: 0.1926201581954956 | VLoss: 82.51680946350098 

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/mimic.py:394: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  action_probs = F.softmax(dist.distribution.logits).cpu().numpy()
WIS estimation: 100%|██████████| 200/200 [00:04<00:00, 44.07it/s]

12.203150575375549


### Imperfect Concept Predictors

In [ ]:
if is_main and run_imperfect:
    results['inaccurate_comparison'] = {}


In [ ]:
if is_main and run_imperfect:
    greedy_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for (func,acc) in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        env, eval_env, additional_info = get_environment(environment_string,[concept_list[i] for i in greedy_idx],seed)

        greedy_inaccurate_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy'] = greedy_inaccurate_reward
        print(greedy_inaccurate_reward)

In [ ]:
if is_main and run_imperfect:
    greedy_iterative_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        
        _, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_iterative_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        greedy_iterative_inaccurate_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed), 
            'concepts': greedy_iterative_idx
        } 

    if greedy_iterative_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy_iterative'] = greedy_iterative_inaccurate_reward
        print(greedy_iterative_inaccurate_reward)

In [ ]:
if is_main and run_imperfect:
    lp_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        lp_inaccurate_reward[modification] = { 'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': lp_idx}

    if lp_inaccurate_reward != {}:
        results['inaccurate_comparison']['lp'] = lp_inaccurate_reward
        print(lp_inaccurate_reward)

In [ ]:
if is_main and run_imperfect:
    imperfect_lp_selection_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        
        if modification == "continuous":
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='min')
        else:
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        imperfect_lp_selection_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

    if imperfect_lp_selection_reward != {}:
        results['inaccurate_comparison']['imperfect_lp'] = imperfect_lp_selection_reward
        print(imperfect_lp_selection_reward)

### Intervention

In [ ]:
if intervention_accuracy_by_concept is not None:
    results['intervention_comparison'] = {}

In [ ]:
if is_main:
    greedy_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        greedy_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_intervention_reward != {}:
        results['intervention_comparison']['greedy'] = greedy_intervention_reward
        print(greedy_intervention_reward)

In [ ]:
if is_main:
    greedy_iterative_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_iterative_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        greedy_iterative_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_iterative_idx
        }

    if greedy_iterative_intervention_reward != {}:
        results['intervention_comparison']['greedy_iterative'] = greedy_iterative_intervention_reward
        print(greedy_iterative_intervention_reward)

In [ ]:
if is_main:
    lp_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        lp_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': lp_idx
        }

    if lp_intervention_reward != {}:
        results['intervention_comparison']['lp'] = lp_intervention_reward
        print(lp_intervention_reward)

In [ ]:
if is_main:
    imperfect_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,intervention_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        subset_concept = [modified_concept_predictors[i] for i in imperfect_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        imperfect_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': imperfect_idx
        }

    if imperfect_intervention_reward != {}:
        results['intervention_comparison']['imperfect_lp'] = imperfect_intervention_reward
        print(imperfect_intervention_reward)

### Iterative

In [ ]:
if is_main and run_iterative:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    results['iterative'] = {}

In [ ]:
if is_main and run_iterative:
    rand_idx = random_selection(concept_list,initial_concepts)[1]    
    rewards_iterative, concepts_iterative = iterative_selection(eval_env,gold_model,environment_string,rand_idx,concept_list,num_iterations,selections_per_round,seed)
    results['iterative']['iterative_selection'] = {'reward': rewards_iterative, 'concepts': concepts_iterative}
    print(rewards_iterative)

In [ ]:
if is_main and run_iterative:
    td_learner = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list,get_td_learner=True)
    rand_idx = random_selection(concept_list,initial_concepts)[1]    
    rewards_td, concepts_td = iterative_selection(eval_env,gold_model,environment_string,rand_idx,concept_list,num_iterations,selections_per_round,seed,td_learner=td_learner)
    results['iterative']['iterative_selection_q'] = {'reward': rewards_td, 'concepts': concepts_td}
    print(rewards_td)

In [ ]:
if is_main and run_iterative:
    num_concepts_selected = num_iterations*selections_per_round+initial_concepts
    bayesian_reward, bayesian_idx = bayesian_iterative_selection(ground_truth_gym_env,environment_string,seed,concept_list,num_iterations,num_concepts_selected)

    results['iterative']['bayesian'] = {
        'reward': bayesian_reward, 
        'concepts': bayesian_idx
    }
    print(bayesian_reward)

### Two-Stage Training

In [ ]:
if is_main and run_two_stage:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,list(range(len(concept_list))))
    results['two_stage'] = {}
    results['two_stage']['accuracy'] = acc_list 


In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_two_stage = {}
    greedy_concepts, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,greedy_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy'] = {'reward': greedy_two_stage_reward, 'concepts': greedy_idx}
    print(greedy_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_iterative_concepts, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,greedy_iterative_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_iterative_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy_iterative'] = {'reward': greedy_iterative_two_stage_reward, 'concepts': greedy_iterative_idx}
    print(greedy_iterative_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    lp_concepts, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,lp_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, lp_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    lp_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['lp'] = {'reward': lp_two_stage_reward, 'concepts': lp_idx}
    print(lp_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    top_k_idx = np.argsort(acc_list)[-num_concepts_selected:]
    top_k_concepts = [concept_list[i] for i in top_k_idx]
    
    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,top_k_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, top_k_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    top_k_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['top_k'] = {'reward': top_k_two_stage_reward, 'concepts': top_k_idx}
    print(top_k_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    imperfect_concepts, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,concept_list,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,acc_list,concept_source,environment_string,additional_info,direction='max')
    
    concept_predictor, _ = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,imperfect_idx)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, imperfect_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    imperfect_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['imperfect'] = {'reward': imperfect_two_stage_reward, 'concepts': imperfect_idx}
    print(imperfect_two_stage_reward)

## Ablations

### Reward Perturbation

In [ ]:
if is_main and reward_error > 0:
    results['reward_error'] = {}
    perturbed_groundtruth_eval_env = RewardPerturbationWrapper(ground_truth_gym_env,reward_error)

    if selection_function == "q_value":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,learning_rate=1e-3,mimic=True,total_timesteps=5000,final_training=0)
        else:
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    elif selection_function == "policy":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,mimic=True)
        else:
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)

    subset_concept, greedy_perturbed_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    perturbed_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,perturbed_model,seed)
    results['reward_error']['greedy'] = {
        'reward': greedy_selection_perturbed_reward,
        'concepts': greedy_perturbed_idx
    }
    print(greedy_selection_perturbed_reward)

In [ ]:
if is_main and reward_error > 0:
    subset_concept, greedy_perturbed_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_iterative_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['greedy_iterative'] = {
        'reward': greedy_iterative_selection_perturbed_reward,
        'concepts': greedy_perturbed_iterative_idx
    }
    print(greedy_iterative_selection_perturbed_reward)

In [ ]:
if is_main and reward_error > 0:
    subset_concept, lp_perturbed_idx = lp_based_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    lp_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['lp'] = {
        'reward': lp_selection_perturbed_reward,
        'concepts': lp_perturbed_idx
    }
    print(lp_selection_perturbed_reward)

### Comparison with Concept Completeness

In [ ]:
# TODO: Create a Shapley-based baseline
if is_main and assess_completeness:
    pass 


## Save Data

In [ ]:
out_folder = "imperfect"

In [ ]:
if is_main:
    save_path = get_save_path(out_folder,save_name)

In [ ]:
if is_main:
    delete_duplicate_results(out_folder,"",results)

In [ ]:
results['two_stage']['top_k']['concepts'] = results['two_stage']['top_k']['concepts'].tolist()

In [ ]:
if is_main:
    json.dump(results,open('../../results/'+save_path,'w'))